# N-Grams: Capturing Short Word Sequences

Extend Bag of Words by preserving limited word order through contiguous word sequences.

> **Project:** SMS spam detection  |  **Technique:** Unigrams, bigrams, and trigrams

## Learning goals

By the end of this notebook, you will be able to:

- define an N-gram and distinguish unigrams, bigrams, and trigrams;
- prepare SMS text for vectorization;
- control N-gram generation with `ngram_range`; and
- inspect how N-grams change the feature space.

### Roadmap
`Setup` → `Load data` → `Prepare text` → `Generate N-grams` → `Compare representations`

## 1. Setup

Install the libraries used in this lesson. If they are already installed in the active environment, this cell can be skipped.

**Libraries:** `pandas` for tabular data, `nltk` for stemming and stop words, and `scikit-learn` for N-gram vectorization.

In [ ]:
%pip install pandas nltk scikit-learn

## 2. Load and inspect the SMS dataset

Each row contains a message and its label: `ham` for a normal message or `spam` for an unwanted message.

> **Path note:** Run this notebook with `Stage 2/03_Text-to-Vectors` as the working directory so the relative dataset path resolves correctly.

In [ ]:
import pandas as pd

DATA_PATH = 'SpamClassifier-master/smsspamcollection/SMSSpamCollection'
messages = pd.read_csv(
    DATA_PATH,
    sep='\t',
    names=['label', 'message'],
)

messages.head()

### Quick data check

Check the number of examples and the label balance before creating features. A class imbalance is important to remember when evaluating a spam classifier.

In [ ]:
print(f'Rows: {len(messages):,}')
print(f'Columns: {list(messages.columns)}')
print('\nClass distribution:')
display(messages['label'].value_counts().rename_axis('label').to_frame('count'))

## 3. Prepare the text

Before generating N-grams, normalize the messages so equivalent words are represented consistently. This pipeline removes punctuation, lowercases text, removes English stop words, and applies Porter stemming.

**Why this matters:** The preprocessing choices affect which sequences become features. For example, removing stop words can eliminate useful phrases such as `not good`, so preprocessing should be tested for the specific task.

In [ ]:
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

nltk.download('stopwords', quiet=True)
english_stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [ ]:
def clean_message(message: str) -> str:
    """Normalize one message before N-gram generation."""
    tokens = re.sub(r'[^a-zA-Z]', ' ', message).lower().split()
    tokens = [
        stemmer.stem(token)
        for token in tokens
        if token not in english_stop_words
    ]
    return ' '.join(tokens)


corpus = messages['message'].map(clean_message)
messages = messages.assign(clean_message=corpus)

messages[['message', 'clean_message']].head()

## 4. What is an N-gram?

An **N-gram** is a contiguous sequence of `N` tokens:

| Type | `N` | Example for `machine learning is useful` |
| --- | ---: | --- |
| Unigram | 1 | `machine`, `learning`, `useful` |
| Bigram | 2 | `machine learning`, `learning useful` |
| Trigram | 3 | `machine learning useful` |

Unlike a basic Bag-of-Words model, N-grams preserve a small amount of local word order. The trade-off is a larger vocabulary and a sparser feature matrix.

In [ ]:
def make_ngrams(tokens: list[str], n: int) -> list[str]:
    """Return contiguous N-grams as space-separated strings."""
    return [' '.join(tokens[index:index + n]) for index in range(len(tokens) - n + 1)]


example_tokens = 'machine learning is useful'.split()
for n in (1, 2, 3):
    print(f'{n}-grams:', make_ngrams(example_tokens, n))

## 5. Generate N-gram features

`CountVectorizer` creates one column per vocabulary item. Set `ngram_range=(1, 1)` for unigrams, `(2, 2)` for only bigrams, or `(1, 3)` to include unigrams, bigrams, and trigrams together.

The matrices below use binary values: `1` means an N-gram appears in a message, while `0` means it does not. `max_features` keeps this demonstration manageable.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

MAX_FEATURES = 2_500

vectorizers = {
    'unigram': CountVectorizer(
        max_features=MAX_FEATURES,
        binary=True,
        ngram_range=(1, 1),
    ),
    'bigram': CountVectorizer(
        max_features=MAX_FEATURES,
        binary=True,
        ngram_range=(2, 2),
    ),
    '1-3 grams': CountVectorizer(
        max_features=MAX_FEATURES,
        binary=True,
        ngram_range=(1, 3),
    ),
}

matrices = {
    name: vectorizer.fit_transform(corpus)
    for name, vectorizer in vectorizers.items()
}

for name, matrix in matrices.items():
    print(f'{name:>10}: shape={matrix.shape}, non-zero values={matrix.nnz:,}')

### Inspect the learned vocabulary

The vocabulary is learned during `.fit_transform()`. It is stored in `vocabulary_` as a mapping from each feature string to its column index.

> **Common mistake:** `vocabulary_` is an attribute, not a function, and it is unavailable until the vectorizer has been fitted.

In [ ]:
for name in ('unigram', 'bigram'):
    vocabulary = vectorizers[name].get_feature_names_out()
    print(f'\n{name.title()} examples:')
    print(list(vocabulary[:15]))

## 6. Compare feature-space size

Adding longer sequences usually creates more possible features. This can help a model recognize phrases, but it can also increase memory use and overfitting risk.

Use the summary below to compare the number of learned features and the sparsity of each representation.

In [ ]:
comparison = pd.DataFrame([
    {
        'representation': name,
        'messages': matrix.shape[0],
        'features': matrix.shape[1],
        'non_zero_values': matrix.nnz,
        'density_percent': round(matrix.nnz / (matrix.shape[0] * matrix.shape[1]) * 100, 3),
    }
    for name, matrix in matrices.items()
])
comparison

## Key takeaways

1. Unigrams represent individual words; bigrams and trigrams represent short phrases.
2. N-grams preserve local word order, which can add useful context to a classifier.
3. Larger `n` values create more features and usually make the matrix sparser.
4. `ngram_range=(1, 3)` combines multiple sequence lengths in one feature matrix.
5. Choose the N-gram range with validation rather than assuming that longer sequences are always better.

### Next step
Train the same classifier with `matrices['unigram']`, `matrices['bigram']`, and `matrices['1-3 grams']`, then compare validation metrics such as precision, recall, and F1-score.